# Day 25 In-Class Assignment: Athena Tables & Thoughtful SQL

**Use this notebook to guide your in-class work on Day 25.**

You will work to:
- Reorganize your S3 data structure for Athena compatibility (we'll demo this together)
- Create Athena external tables pointing to your NYC 311 CSV files
- Write a sequence of SQL queries exploring the data, including a "stakeholder question" query
- Identify and fix at least one "tricky" query
- Document your sanity-checks

**At the end of class, commit your `.sql` files and `notes/sanity_check_log.md` to GitHub and submit this notebook to D2L.** Details for this are include below!

> **Name (FIRST and LAST):** _delete this text and type your name here before submitting this notebook_
>
> **Your S3 bucket (for easy of reference; use the _full_ bucket name, including the account regional suffix):**  
> `s3://___`
>
> **Your GitHub repo URL (from the Day 24 class period):**  
> `https://github.com/___/___`

- - -

## Learning objectives

By the end of this class period, you should be able to:

- **Reorganize S3 data** into separate prefixes for Athena compatibility.
- **Create Athena external tables** that point to CSV data stored in S3.
- **Write SQL queries** (SELECT, FROM, WHERE, GROUP BY, JOIN) against Athena tables.
- **Identify implausible results** and refine queries to fix issues.
- **Document sanity-checks** so you (and teammates) can understand your reasoning.
- **Tie query results back** to a stakeholder/problem brief.

- - -

## Table of contents

1. [S3 data reorganization](#reorganize)
2. [Athena setup and database](#setup)
3. [Creating external tables](#tables)
4. [Warm-up queries](#warmup)
5. [Tricky/edge-case query](#tricky)
6. [Sanity-check log](#sanity)
7. [Git commit and push](#git)

- - -

<a id="reorganize"></a>

## 1. S3 data reorganization

As a class, we'll take a minute to reorganize our S3 data structure to be compatible with Athena's requirements for external tables. The reason for and nature of this change is included here as a reference, but **unless you miss class or weren't able to follow along and do it in real time, you should be able to skip this section and move on to the next one.**

### Why reorganize?

In Athena, the `LOCATION` clause points to a **folder** (S3 prefix), and Athena reads **all files** in that folder (as I discovered when testing out the creation of external tables!). So, if you point to `s3://bucket/raw/`, it will try to read both `complaints.csv` and `agencies.csv` into the same table—which is not what we want!

**Solution:** Reorganize into separate prefixes:
```
s3://your-bucket/raw/
├── complaints/
│   └── complaints.csv
└── agencies/
    └── agencies.csv
```

This way, each Athena table can point to its own prefix.

### Steps to reorganize

**Option 1: AWS Console (easiest)**

1. Open S3 console → Your bucket → `raw/` folder
2. Create new folders: `complaints/` and `agencies/`
3. Move (under Actions) `complaints.csv` into the `complaints/` folder (you'll need the full S3 path you want to move it to, e.g. `s3://your-bucket/raw/complaints/`)
4. Move `agencies.csv` into the `agencies/` folder (similarly, you'll need the full S3 path, e.g. `s3://your-bucket/raw/agencies/`)

**Option 2: AWS Command Line Interface (CLI)**

You can also use the AWS CLI to move files, you can access the CLI by going to `CloudShell` in the AWS console (you can search for it as a service or select it from the bottom left of the console).

The `aws s3 mv` command allows you to move files within S3. Here's how you can do it:

```bash
aws s3 mv s3://YOUR-BUCKET/raw/complaints.csv s3://YOUR-BUCKET/raw/complaints/complaints.csv
aws s3 mv s3://YOUR-BUCKET/raw/agencies.csv s3://YOUR-BUCKET/raw/agencies/agencies.csv
```

(Replace `YOUR-BUCKET` with your actual _full_ bucket name, including the regional suffix, e.g. `my-bucket-123456789012-us-east-1-an`)

### Verification

In S3 console:
- Navigate to `raw/` folder
- You should see two subfolders: `complaints/` and `agencies/`
- Click each to verify the CSV is inside

- - -

<a id="setup"></a>

## 2. Athena setup and database

Again, we did this section together in class, so **if you were able to follow along and do it in real time, you should be able to skip this section and move on to the next one.** The rest should serve as a reference if you need to set up Athena on your own.

### Accessing  Athena

1. AWS Console → Search "Athena" → Click "Athena"
2. You'll see the Athena query editor
3. Use the default workgroup (should "primary")

### Set query result location

Before running queries, Athena needs a place to store results:

1. Click the **"QuerySettings"** tab
2. Look for "Query Result Encryption"
3. Click **"Manage"** and set:
   - **Query Result Location:** `s3://YOUR-BUCKET/athena-results/`

### Create the database

To initialize the database, in the Athena query editor (main text box), copy and run:

```sql
CREATE DATABASE IF NOT EXISTS nyc311_db;
```

Click the **"Run query"** button (or press Ctrl+Enter).

You should see a success message like "Query succeeded."

- - -

<a id="tables"></a>

## 3. Creating external tables

Now that you have the database set up and your data organized in S3, you need to create the external tables that point to your CSV files. This will allow you to run SQL queries against the data in Athena.

### Table 1: Complaints

First, you'll create an external table for complaints.

In a new query tab, copy and run (update `YOUR-BUCKET` in the `LOCATION` clause below with your actual bucket name):

```sql
CREATE EXTERNAL TABLE IF NOT EXISTS nyc311_db.complaints (
  unique_key             int,
  created_date           string,
  closed_date            string,
  agency                 string,
  agency_name            string,
  problem                string,
  problem_detail         string,
  incident_zip           string,
  status                 string,
  resolution_description string,
  borough                string
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
  'separatorChar' = ',',
  'quoteChar'     = '\"'
)
LOCATION 's3://YOUR-BUCKET/raw/complaints/' -- Update this to match your actual S3 path!
TBLPROPERTIES ('skip.header.line.count'='1');
```

**Key points about this query:**
- Replace `YOUR-BUCKET` with your actual S3 bucket name.
- `LOCATION` should point to the `complaints/` **folder/prefix**, not the `.csv` file. This allows Athena to read the file properly.
- Column names and types must match your data dictionary exactly
- `skip.header.line.count='1'` tells Athena to skip the CSV header row
- The `SERDE` and `SERDEPROPERTIES` handle CSV parsing, including quoted fields and commas inside quotes. `SERDE` stands for "Serializer/Deserializer" and tells Athena how to read the data format. You'll notice that SERDE is using `OpenCSVSerde`, which is a common choice for CSV files, though not the only option. If you're curious about this piece, you can read more about Athena's SerDes here: https://docs.aws.amazon.com/athena/latest/ug/supported-serdes.html

Click **"Run query"** → you should see "Query succeeded." Additionally, now, in the left sidebar, you should be able to see `complaints` listed under `nyc311_db` tables

### Table 2: Agencies

Similar to the complaints table, now you'll create an external table for agencies.

In the same or a new query tab (click the "+" button to create a new tab), run:

```sql
CREATE EXTERNAL TABLE IF NOT EXISTS nyc311_db.agencies (
  agency      string,
  agency_name string
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
  'separatorChar' = ',',
  'quoteChar'     = '\"'
)
LOCATION 's3://YOUR-BUCKET/raw/agencies/' -- Update this to match your actual S3 path!
TBLPROPERTIES ('skip.header.line.count'='1');
```

Click **"Run query"** → should succeed. Again, you should also now see `agencies` listed under `nyc311_db` tables in the left sidebar.

### Checkpoint query

Run a quick test to confirm tables work:

```sql
SELECT * FROM nyc311_db.agencies LIMIT 5;
```

You should see the agency data (a few rows). If you get an error, check:
- S3 path is correct
- CSV files are in the right folders (`raw/complaints/`, `raw/agencies/`)
- Column names and types match the data

- - -

<a id="warmup"></a>

## 4. Part A: Warm-up queries

### Overview

You'll now write a series of queries to explore the NYC 311 data. To confirm that everything is working as expected.

As you work through each of these queries:

1. Review the query
2. Predict what it will produce
3. Run it within Athena and confirm that it produces the expected result
4. Once successful, copy it into your `sql/` folder within your repository within a `.sql` file (e.g., `sql/warm_ups.sql`). Include a brief comment above each query in the file describing what it does and what the expected result is. Query 1 provides an example of how to do this.

### Query 1: Row count

```sql
SELECT COUNT(*) AS n_complaints
FROM nyc311_db.complaints;
```

Assuming the above query works, you would then copy this query into your `warm_ups.sql` file with in your `sql/` folder within your repository with a comment describing what it does and what the expected result is. For example:

```sql
-- This query counts the total number of complaints in the dataset. The expected result is 200,000 (as a single number).
SELECT COUNT(*) AS n_complaints
FROM nyc311_db.complaints;
```

### Query 2: Date range

```sql
SELECT 
  MIN(created_date) AS earliest,
  MAX(created_date) AS latest
FROM nyc311_db.complaints;
```

### Query 3: Complaints by agency

```sql
SELECT agency, COUNT(*) AS n
FROM nyc311_db.complaints
GROUP BY agency
ORDER BY n DESC
LIMIT 10;
```

### Query 4: Complaint types by borough

```sql
SELECT borough, problem, COUNT(*) AS n
FROM nyc311_db.complaints
GROUP BY borough, problem
ORDER BY n DESC
LIMIT 20;
```

### Query 5: Join with agencies table

```sql
SELECT 
  c.agency,
  a.agency_name,
  COUNT(*) AS n
FROM nyc311_db.complaints AS c
JOIN nyc311_db.agencies AS a
  ON c.agency = a.agency
GROUP BY c.agency, a.agency_name
ORDER BY n DESC;
```

---

### You turn: Write a stakeholder-focused query

Thinking about to the **problem brief** idea we discussed on Day 23. Come up with a "stakeholder question" and write a query to answer it. For example:
- "Which borough has the most noise complaints?"
- "How do complaint volumes vary by agency and problem type?"

You can use the above queries as a starting point and tweak them to align with your stakeholder question. For example, if your stakeholder is interested in heat/hot water complaints, you could filter the complaint types to `WHERE problem = 'HEAT/HOT WATER'`.

Save your query as a new `.sql` file, `sql/stakeholder_query.sql`, and note the question and expected results.

- - -

<a id="tricky"></a>

## 5. Part B: Tricky/buggy query

This part presents a query that seeks to answer a seemingly straightforward question, but has an issue that leads to error.

**The goal is to determine what the query is trying to do, identify the issue, understand why it happens, and refine the query to fix it.**

### Average resolution time (buggy approach)

```sql
SELECT
  agency,
  AVG(
    date_diff(
      'day',
      date_parse(created_date, '%Y-%m-%d %H:%i:%s'),
      date_parse(closed_date,  '%Y-%m-%d %H:%i:%s')
    )
  ) AS avg_days_to_close
FROM nyc311_db.complaints
GROUP BY agency
ORDER BY avg_days_to_close DESC;
```

**Run it and observe:** Does it error? What is causing the error?

### Creating a refined/improved version

Once you've been able to identify the issue with the above query, you can solve it in different ways. Come up with a solution that you think makes the most sense, and implement it in a new `.sql` file: `sql/resolution_time.sql`. Leave a comment at the top describing the issue you identified and your reasoning for how you chose to fix it.

### Your findings

Reflect on what you learned from this exercise:

> **5.4 Reflection on buggy query**
>
> - What assumption(s) did the original query make?  
>   _Your answer:_
>
> - What did you have to change to make run? How did you justify that change?  
>   _Your answer:_
>
> - Does your revised query make sense as an answer to a possible stakeholder question? Why?  
>   _Your answer:_

- - -

<a id="sanity"></a>

## 6. Sanity-check log

### What is it?

Now that you've run some queries and debugged one, you're going to create a **sanity-check log**.

When you run into unexpected results (whether it's an error or just a surprising output), it can be important to document your thought process and checks. Your sanity-check log is a document where you record:
- The query you ran
- What you expected to see
- What you actually saw
- Any checks or refinements you did
- Whether the result makes sense

This is valuable for:
- **Reproducibility:** Future you (or teammates) can see your reasoning
- **Model validation:** Later, you might use similar thinking when evaluating SageMaker models
- **Stakeholder trust:** Showing you've validated results before reporting them

### Create your log

In your project repo, create or open:

**File:** `notes/sanity_check_log.md`

Add an entry like based on this template (filling in the details based on your experience with the tricky query you just worked through):

```markdown
## Query: Average resolution time by agency (2026-03-27)

- **File:** sql/resolution_time.sql
- **Business question:** How long does each agency take to resolve complaints?
- **What I expected:** ... (this could be something about the nature of the results, e.g. "I expected HPD and DEP to be faster than NYPD due to the nature of their work.")
- **Issues encountered:** ... (describe issues you ran into)
- **Checks performed:** ... (describe the checks you did to validate results and/or debug the query)
- **Final outcome:** ... (describe whether the final results are plausible, what you learned from the process, and if any further refinements are needed)
- **Confidence:** High. Would present to stakeholder.
```

**Make sure you save the files after adding this entry!**

- - -

<a id="git"></a>

## 7. Git commit and push

### Prepare your local repo

In your project root (`aws-nyc311-yourMSUNetID/` or wherever your repo is located), verify you have:
- `sql/` folder with new `.sql` files
- `notes/sanity_check_log.md` with your first sanity-check entry

### Stage, commit, and push

Make sure you add both your new SQL files and your sanity-check log to the repo and then commit, with a descriptive message about what you added/changed.

Then, push to GitHub!

### Verify on GitHub

Go to GitHub.com → Your repo → You should see:
- New/updated files in `sql/` folder
- Updated `notes/sanity_check_log.md`
- Commit message in history

- - -

### Congratulations, you're done!

Submit this assignment by uploading it to the course Desire2Learn web page. Go to the In-class assignments section, find the appropriate submission link, and upload it there.

See you in class!

---
© Copyright 2026, The Department of Computational Mathematics, Science and Engineering at Michigan State University.